In [ ]:
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
pd.set_option('display.max_columns', None)

In [ ]:
# Define the base path
base_path = "./data-all-waves"
file_path = os.path.join(base_path, "df_final.csv")
df_final = pd.read_csv(file_path)

In [ ]:
df_final.columns

In [ ]:
# Function to clean EDMOM and EDDAD values
def clean_ed(x):
    if pd.isna(x):  # Keep NaN as is
        return x
    if isinstance(x, str):
        if x.isdigit():  # Convert string numbers to int
            return int(x)
        elif x == 'E':   # Keep 'E' as is
            return 'E'
    return x  # Return original value if it's already an integer

# Apply the cleaning function to EDMOM and EDDAD for all years
df_final['EDMOM'] = df_final['EDMOM'].apply(clean_ed)
df_final['EDDAD'] = df_final['EDDAD'].apply(clean_ed)

# Verify the cleaned values
for year in sorted(df_final['REFYR'].unique()):
    edmom_values = df_final[df_final['REFYR'] == year]['EDMOM'].unique()
    eddad_values = df_final[df_final['REFYR'] == year]['EDDAD'].unique()
    
    print(f"\nREFYR {year}:")
    print(sorted([x for x in edmom_values if x != 'E']) + (['E'] if 'E' in edmom_values else []))
    print(sorted([x for x in eddad_values if x != 'E']) + (['E'] if 'E' in eddad_values else []))


In [ ]:
import os
import pandas as pd

# Define the path and read the CSV file
base_path = "./data-all-waves"
ur_ba_path = os.path.join(base_path, "ur_ba_16y.csv")

# Read the CSV file
ur_ba_16y = pd.read_csv(ur_ba_path)

# Rename columns
ur_ba_16y = ur_ba_16y.rename(columns={'CGBD16O': 'UR'})

# First prepare yearly unemployment rates for both degree types
# For Bachelor's
ur_ba_16y['observation_date'] = pd.to_datetime(ur_ba_16y['observation_date'])
ur_ba_16y['year'] = ur_ba_16y['observation_date'].dt.year


# Assume there is a 'YEAR' column in ur_ba_16y indicating the year
# If not, check if 'BAYR' corresponds to a year column
if 'year' in ur_ba_16y.columns:
    # Compute the annual average unemployment rate
    ur_avg = ur_ba_16y.groupby('year')['UR'].mean().reset_index()
    
    # Rename YEAR to BAYR so it matches df_final
    ur_avg = ur_avg.rename(columns={'year': 'BAYR'})
else:
    print("Error: No 'year' column found in ur_ba_16y. Please check the dataset structure.")

# Merge with df_final based on BAYR
df_final= df_final.merge(ur_avg, on='BAYR', how='left')

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from statsmodels.discrete.discrete_model import Probit

def prepare_data(df):
    """
    Prepare data for regression analysis
    """
    # Keep only the last observation for each person
    df = df.sort_values('REFYR').drop_duplicates(subset='assigned_id', keep='last')
    # Create immediate masters indicator
    df['immediate_masters'] = ((df['DGRDG'] == 2) & 
                             (df['DGRYR'] - df['BAYR'] <= 3)).astype(int)
    df['with_masters'] = (df['DGRDG'] == 2 ).astype(int)
    
    # Create cohort trend (centered)
    df['cohort_trend'] = df['BAYR'] - df['BAYR'].min()  # Linear trend
    df['cohort_trend_sq'] = df['cohort_trend']**2       # Quadratic trend
    df['cohort_trend_cub'] = df['cohort_trend']**3       # Quadratic trend
    
    # Create dummy variables for categorical variables
    gender_dummies = pd.get_dummies(df['GENDER'], prefix='gender')
    race_dummies = pd.get_dummies(df['RACEM'], prefix='race')
    major_dummies = pd.get_dummies(df['NBAMEMG'], prefix='major')
    
    # Print diagnostic information
    print("Sample information:")
    print(f"Number of unique individuals: {df['assigned_id'].nunique()}")
    print(f"Year range: {df['REFYR'].min()} - {df['REFYR'].max()}")
    print(f"Bachelor graduation year range: {df['BAYR'].min()} - {df['BAYR'].max()}")
    print("\nDistribution of immediate_masters:")
    print(df['immediate_masters'].value_counts(normalize=True))
    print("\nUnemployment rate summary:")
    print(df['gradur_ba'].describe())
    
    return df

def run_probit_analysis(df, formula):
    """
    Run probit regression using formula interface
    """
    model = smf.probit(formula, data=df)
    results = model.fit(cov_type='HC1')
    
    return results

def run_probit_analysis_robust_cohort(df, formula):
    """
    Run probit regression using formula interface
    """
    
    model = smf.probit(formula, data=df)
   # Cluster standard errors at the graduation cohort level
    results = model.fit(cov_type='cluster', 
                       cov_kwds={'groups': df['BAYR']})
    
    return results



In [ ]:
# To use:
df = prepare_data(df_final)

df = df[df['BAYR']< 2020]


formula_basic = '''
with_masters ~ UR + cohort_trend + cohort_trend_sq + C(GENDER) + AGE + C(RACEM) + C(EDMOM) + C(EDDAD)
'''

results_basic_1 = run_probit_analysis_robust_cohort(df, formula_basic)
print(results_basic_1.summary())



In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from statsmodels.discrete.discrete_model import Probit

def prepare_data(df):
    """
    Prepare data for regression analysis
    """
    # Keep only the last observation for each person
    df = df.sort_values('REFYR').drop_duplicates(subset='assigned_id', keep='last')
    # Create immediate masters indicator
    df['immediate_masters'] = ((df['DGRDG'] == 2) & 
                             (df['DGRYR'] - df['BAYR'] <= 3)).astype(int)
    df['with_masters'] = (df['DGRDG'] == 2 ).astype(int)
    
    # Create cohort trend (centered)
    df['cohort_trend'] = df['BAYR'] - df['BAYR'].min()  # Linear trend
    df['cohort_trend_sq'] = df['cohort_trend']**2       # Quadratic trend
    df['cohort_trend_cub'] = df['cohort_trend']**3       # Quadratic trend
    
    # Create dummy variables for categorical variables
    gender_dummies = pd.get_dummies(df['GENDER'], prefix='gender')
    race_dummies = pd.get_dummies(df['RACEM'], prefix='race')
    major_dummies = pd.get_dummies(df['NBAMEMG'], prefix='major')
    
    # Print diagnostic information
    print("Sample information:")
    print(f"Number of unique individuals: {df['assigned_id'].nunique()}")
    print(f"Year range: {df['REFYR'].min()} - {df['REFYR'].max()}")
    print(f"Bachelor graduation year range: {df['BAYR'].min()} - {df['BAYR'].max()}")
    print("\nDistribution of immediate_masters:")
    print(df['immediate_masters'].value_counts(normalize=True))
    print("\nUnemployment rate summary:")
    print(df['gradur_ba'].describe())
    
    return df

def run_probit_analysis(df, formula):
    """
    Run probit regression using formula interface
    """
    model = smf.probit(formula, data=df)
    results = model.fit(cov_type='HC1')
    
    return results

def run_probit_analysis_robust_cohort(df, formula):
    """
    Run probit regression using formula interface with robust standard errors
    """
    model = smf.probit(formula, data=df)
    # Cluster standard errors at the graduation cohort level
    results = model.fit(cov_type='cluster', 
                       cov_kwds={'groups': df['BAYR']})
    
    return results

def run_gender_specific_analysis(df, formula_base):
    """
    Run separate analyses for males and females
    
    Parameters:
    -----------
    df: DataFrame containing the prepared data
    formula_base: Base formula string without gender controls
    
    Returns:
    --------
    dict: Dictionary containing results for all, males, and females
    """
    results = {}
    
    # Assuming GENDER=1 is male and GENDER=2 is female
    # Verify this coding in your dataset and adjust if needed
    male_df = df[df['GENDER'] == 'M']
    female_df = df[df['GENDER'] == 'F']
    
    # Remove gender from formula for gender-specific regressions
    gender_specific_formula = formula_base.replace("+ C(GENDER)", "")
    
    print(f"\n==== OVERALL ANALYSIS (n={len(df)}) ====")
    results['all'] = run_probit_analysis_robust_cohort(df, formula_base)
    print(results['all'].summary())
    
    print(f"\n==== MALE ANALYSIS (n={len(male_df)}) ====")
    results['male'] = run_probit_analysis_robust_cohort(male_df, gender_specific_formula)
    print(results['male'].summary())
    
    print(f"\n==== FEMALE ANALYSIS (n={len(female_df)}) ====")
    results['female'] = run_probit_analysis_robust_cohort(female_df, gender_specific_formula)
    print(results['female'].summary())
    
    return results

# To use:
df = prepare_data(df_final)
df = df[df['BAYR'] < 2020]

# Define the base formula
formula_basic = '''
with_masters ~ UR + cohort_trend + cohort_trend_sq + C(GENDER) + AGE + C(RACEM) + C(EDMOM) + C(EDDAD)
'''

# Run gender-specific analyses
results_dict = run_gender_specific_analysis(df, formula_basic)

# The results can now be accessed via results_dict['all'], results_dict['male'], results_dict['female']
# For example, to compare the marginal effect of UR across genders:
print("\n==== COMPARING UR COEFFICIENTS ACROSS GENDERS ====")
print(f"All:    {results_dict['all'].params['UR']:.4f} (p={results_dict['all'].pvalues['UR']:.4f})")
print(f"Males:  {results_dict['male'].params['UR']:.4f} (p={results_dict['male'].pvalues['UR']:.4f})")
print(f"Females: {results_dict['female'].params['UR']:.4f} (p={results_dict['female'].pvalues['UR']:.4f})")

In [ ]:
df['GENDER'].unique()

In [ ]:
def plot_master_pursuit_by_ur(data, cohort_column='BAYR', ur_column='UR', master_column='with_masters'):
    """
    Plot initial unemployment rate and master's degree pursuit rate by cohort.
    
    Parameters:
    -----------
    data : pandas DataFrame
        The dataset containing cohort information, unemployment rates, and master status
    cohort_column : str
        Column name for graduation cohort year
    ur_column : str
        Column name for unemployment rate at graduation
    master_column : str
        Column name for master's degree indicator (1=yes, 0=no)
    """
    import matplotlib.pyplot as plt
    import pandas as pd
    import numpy as np
    
    # Group data by cohort and calculate statistics
    cohort_stats = data.groupby(cohort_column).agg({
        ur_column: 'mean',            # Average UR for each cohort  
        master_column: 'mean'         # Share with master's degree
    }).reset_index()
    
    # Sort by cohort year
    cohort_stats = cohort_stats.sort_values(by=cohort_column)
    
    # Create figure with two y-axes
    fig, ax1 = plt.figure(figsize=(12, 6)), plt.gca()
    ax2 = ax1.twinx()
    
    # Plot unemployment rate on left axis
    line1 = ax1.plot(cohort_stats[cohort_column], cohort_stats[ur_column], 
                    'r-', marker='o', linewidth=2, label='Unemployment Rate')
    ax1.set_xlabel('Bachelor Graduation Year')
    ax1.set_ylabel('Unemployment Rate (%)', color='blue')
    ax1.tick_params(axis='y', labelcolor='blue')
    
    # Plot master's share on right axis
    line2 = ax2.plot(cohort_stats[cohort_column], cohort_stats[master_column]*100, 
                    'b-', marker='s', linewidth=2, label='Master\'s Degree (%)') 
    ax2.set_ylabel('Share with Master\'s Degree (%)', color='red')
    ax2.tick_params(axis='y', labelcolor='red')
    
    # Calculate correlation
    correlation = np.corrcoef(cohort_stats[ur_column], cohort_stats[master_column])[0,1]
    
    # Add correlation to the plot
    plt.annotate(f"Correlation: {correlation:.3f}", 
                 xy=(0.05, 0.95), 
                 xycoords='axes fraction',
                 bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray", alpha=0.8))
    
    # Add legend
    lines = line1 + line2
    labels = [l.get_label() for l in lines]
    ax1.legend(lines, labels, loc='upper center', bbox_to_anchor=(0.5, -0.1), 
               ncol=2, frameon=True)
    
    # Finalize plot
    plt.title('Unemployment Rate and Master\'s Degree Pursuit by Graduation Cohort')
    plt.grid(True, alpha=0.3)
    
    # Add recession bars for context if you have that data
    # ax1.fill_between([2001, 2002], 0, 100, alpha=0.2, color='gray')
    # ax1.fill_between([2007, 2009], 0, 100, alpha=0.2, color='gray')
    
    fig.tight_layout()
    return fig

# Usage example:
fig = plot_master_pursuit_by_ur(df)
plt.show()

In [ ]:
def plot_master_pursuit_by_ur(data, cohort_column='BAYR', ur_column='UR', master_column='with_masters'):
    """
    Plot initial unemployment rate and master's degree pursuit rate by cohort.
    
    Parameters:
    -----------
    data : pandas DataFrame
        The dataset containing cohort information, unemployment rates, and master status
    cohort_column : str
        Column name for graduation cohort year
    ur_column : str
        Column name for unemployment rate at graduation
    master_column : str
        Column name for master's degree indicator (1=yes, 0=no)
    """
    import matplotlib.pyplot as plt
    import pandas as pd
    import numpy as np
    
    # Group data by cohort and calculate statistics
    cohort_stats = data.groupby(cohort_column).agg({
        ur_column: 'mean',            # Average UR for each cohort  
        master_column: 'mean'         # Share with master's degree
    }).reset_index()
    
    # Sort by cohort year
    cohort_stats = cohort_stats.sort_values(by=cohort_column)
    
    # Create figure with two y-axes
    fig, ax1 = plt.subplots(figsize=(12, 6))
    ax2 = ax1.twinx()
    
    # Plot unemployment rate on left axis
    line1 = ax1.plot(cohort_stats[cohort_column], cohort_stats[ur_column], 
                    'r-', marker='o', linewidth=2, label='Unemployment Rate')
    ax1.set_xlabel('Bachelor Graduation Year')
    ax1.set_ylabel('Unemployment Rate (%)', color='blue')
    ax1.tick_params(axis='y', labelcolor='blue')
    
    # Set x-axis to show integer years
    ax1.set_xticks(cohort_stats[cohort_column])
    ax1.set_xticklabels([int(year) for year in cohort_stats[cohort_column]], rotation=45)
    
    # Plot master's share on right axis
    line2 = ax2.plot(cohort_stats[cohort_column], cohort_stats[master_column]*100, 
                    'b-', marker='s', linewidth=2, label='Master\'s Degree (%)') 
    ax2.set_ylabel('Share with Master\'s Degree (%)', color='red')
    ax2.tick_params(axis='y', labelcolor='red')
    
    # Calculate correlation
    correlation = np.corrcoef(cohort_stats[ur_column], cohort_stats[master_column])[0,1]
    
    # Add correlation to the plot
    plt.annotate(f"Correlation: {correlation:.3f}", 
                 xy=(0.05, 0.95), 
                 xycoords='axes fraction',
                 bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray", alpha=0.8))
    
    # Add legend
    lines = line1 + line2
    labels = [l.get_label() for l in lines]
    ax1.legend(lines, labels, loc='upper center', bbox_to_anchor=(0.5, -0.15), 
               ncol=2, frameon=True)
    
    # Finalize plot
#     plt.title('Unemployment Rate and Master\'s Degree Attainment by Graduation Cohort')
    plt.grid(True, alpha=0.3)
    
    fig.tight_layout()
    return fig
# Usage example:
fig = plot_master_pursuit_by_ur(df)
plt.show()

# Composition of Master's degree holders

In [ ]:
def run_direct_regression_analysis(df):
    """
    Run direct regressions of composition characteristics on UR
    """
    # Keep only master's degree holders
    masters_df = df[df['with_masters'] == 1].copy()
    
    # Create characteristics to analyze
    masters_df['is_female'] = (masters_df['GENDER'] == 'F').astype(int)
    masters_df['age_at_masters'] = masters_df['DGRYR'] - masters_df['BIRYR']
    masters_df['ba_to_masters_years'] = masters_df['DGRYR'] - masters_df['BAYR']
    
    # Create different racial group indicators
    # (Create appropriate dummy variables based on your RACEM coding)
    # Example assuming RACEM has common codes:
    masters_df['is_white'] = (masters_df['RACEM'] == 4).astype(int)
    masters_df['is_black'] = (masters_df['RACEM'] == 3).astype(int)
    masters_df['is_asian'] = (masters_df['RACEM'] == 1).astype(int)
    
    # List of characteristics to analyze
    characteristics = [
        'is_female', 
        'is_white', 
        'is_black', 
        'is_asian', 
        'age_at_masters', 
        'ba_to_masters_years'
    ]
    
    # Run regressions for each characteristic
    regression_results = {}
    for char in characteristics:
        # Basic formula with cohort trends
        formula = f"{char} ~ UR + cohort_trend + cohort_trend_sq"
        
        # Run regression
        result = smf.ols(formula, data=masters_df).fit(cov_type='cluster', 
                       cov_kwds={'groups': masters_df['BAYR']})
        regression_results[char] = result
        
        # Print summary for each characteristic
        print(f"\n\n--- {char} Regression Results ---")
        print(result.summary().tables[1])  # Print only coefficients table
    
    return regression_results

regression_results = run_direct_regression_analysis(df)

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import seaborn as sns

def analyze_masters_composition(df):
    """
    Analyze how the composition of master's degree holders varies with initial unemployment rate
    """
#     # Keep only the last observation for each person
#     df = df.sort_values('REFYR').drop_duplicates(subset='assigned_id', keep='last')
    
    # Filter to only include people with master's degrees
    masters_df = df[df['with_masters'] == 1].copy()
    
    # Create UR bins for analysis
    masters_df['ur_bin'] = pd.qcut(masters_df['UR'], 4, labels=['Low', 'Medium-Low', 'Medium-High', 'High'])
    
    # Results dictionary to store all our analysis
    results = {}
    
    # 1. Gender composition
    # Create binary gender indicator (assuming GENDER=2 is female)
    masters_df['is_female'] = (masters_df['GENDER'] == 'F').astype(int)
    
    gender_comp = masters_df.groupby('ur_bin')['is_female'].mean().reset_index()
    gender_comp.columns = ['ur_bin', 'female_proportion']
    results['gender'] = gender_comp
    
    # Run regression to test if the relationship is statistically significant
    gender_reg = smf.ols('is_female ~ UR + cohort_trend + cohort_trend_sq', data=masters_df).fit(cov_type='cluster', 
                       cov_kwds={'groups': masters_df['BAYR']})
    results['gender_reg'] = gender_reg
    
    # 2. Race composition
    race_comp = masters_df.groupby('ur_bin')['RACEM'].value_counts(normalize=True).unstack()
    results['race'] = race_comp
    
    # Run regressions for each major racial group
    # Create dummy variables for major racial groups
    race_dummies = pd.get_dummies(masters_df['RACEM'], prefix='race')
    masters_df = pd.concat([masters_df, race_dummies], axis=1)
    
    # We'll run a regression for each major racial category
    race_cols = race_dummies.columns
    race_regs = {}
    for race_col in race_cols:
        race_reg = smf.ols(f'{race_col} ~ UR + cohort_trend + cohort_trend_sq', data=masters_df).fit(cov_type='cluster', 
                       cov_kwds={'groups': masters_df['BAYR']})
        race_regs[race_col] = race_reg
    
    results['race_regs'] = race_regs
    
    # 3. Age distribution
    # Calculate age at master's degree
    masters_df['age_at_masters'] = masters_df['DGRYR'] - masters_df['BIRYR']
    
    # Average age by UR bin
    age_comp = masters_df.groupby('ur_bin')['age_at_masters'].agg(['mean', 'std', 'count'])
    results['age'] = age_comp
    
    # Age regression
    age_reg = smf.ols('age_at_masters ~ UR + cohort_trend + cohort_trend_sq', data=masters_df).fit(cov_type='cluster', 
                       cov_kwds={'groups': masters_df['BAYR']})
    results['age_reg'] = age_reg
    
    # 4. Field of study composition
    # Using NDGMEMG for master's degree field
    field_comp = masters_df.groupby('ur_bin')['NDGMEMG'].value_counts(normalize=True).unstack()
    results['field'] = field_comp
    
    # Create dummy variables for fields
    field_dummies = pd.get_dummies(masters_df['NDGMEMG'], prefix='field')
    masters_df = pd.concat([masters_df, field_dummies], axis=1)
    
    # Run regressions for each major field
    field_cols = field_dummies.columns
    field_regs = {}
    for field_col in field_cols:
        field_reg = smf.ols(f'{field_col} ~ UR + cohort_trend + cohort_trend_sq', data=masters_df).fit(cov_type='cluster', 
                       cov_kwds={'groups': masters_df['BAYR']})
    
    results['field_regs'] = field_regs
    # Add the return statement to return both results and the DataFrame
    return results, masters_df



In [ ]:
def visualize_composition_results(results):
    """
    Create visualizations for composition analysis results
    """
    # 1. Gender Composition Plot
    plt.figure(figsize=(10, 6))
    plt.bar(results['gender']['ur_bin'], results['gender']['female_proportion'])
#     plt.title('Proportion of Female Master\'s Holders by Initial Unemployment Rate')
    plt.ylabel('Proportion')
    plt.xlabel('Unemployment Rate Bin', fontsize=12)
    plt.xticks(rotation=0)  # Horizontal labels
    plt.tight_layout()
    plt.savefig('gender_composition.png', dpi=300)
    plt.show()
    
    # 2. Race Composition Plot
    if 'race' in results:
        plt.figure(figsize=(12, 8))  # Wide figure
        
        # The race data is already indexed by ur_bin with race categories as columns
        ax = results['race'].plot(kind='bar', stacked=True)
        
        # Add race category labels (you may want to customize these based on your codebook)
        race_labels = {
            1: 'Asian ONLY', 
            2: 'American Indian/Alaska Native ONLY', 
            3: 'Black ONLY', 
            4: 'White ONLY', 
            5: 'Native Hawaiian/Other Pacific Islander ONLY', 
            6: 'Multiple Race'
        }
        
#         plt.title('Racial Composition by Initial Unemployment Rate', fontsize=14)
        plt.ylabel('Proportion', fontsize=12)
        
        plt.xlabel('Unemployment Rate Bin', fontsize=10)
        
        # Make x-axis labels horizontal
        plt.xticks(rotation=0)
        
        # Add a bit more padding between x-axis labels and legend
        plt.subplots_adjust(bottom=0.3)  
        
        # Move legend to the bottom outside the plot with smaller font size
        plt.legend(
            title='Race', 
            labels=[race_labels.get(i, f'Race {i}') for i in results['race'].columns],
            bbox_to_anchor=(0.5, -0.25),  # Move legend further down
            loc='upper center',  # Anchor at upper center
            fontsize=8,  # Smaller font size
            title_fontsize=10,  # Smaller title font size
            ncol=3  # Use multiple columns for the legend
        )
        
        plt.tight_layout()
        
        plt.savefig('race_composition.png', dpi=300)
        plt.show()
    
    # 3. Age Distribution
    if 'age' in results:
        plt.figure(figsize=(10, 6))
        results['age']['mean'].plot(kind='bar')
#         plt.title('Average Age at Master\'s Degree by Initial Unemployment Rate')
        plt.ylabel('Age')
        plt.xlabel('Unemployment Rate Bin', fontsize=12)

        plt.xticks(rotation=0)  # Horizontal labels
        plt.tight_layout()
        plt.savefig('age_distribution.png', dpi=300)
        plt.show()

    # 4. Field of Study Composition Plot
    if 'field' in results:   
        # Field code to description mapping based on common classifications
        field_labels = {
            1: 'Computer and Mathematical Sciences',
            2: 'Life Sciences',
            3: 'Physical Sciences',
            4: 'Social Sciences',
            5: 'Engineering',
            6: 'S&E-Related Fields',
            7: 'Non-S&E Fields',

        }
        
        # Handle various possible structures
        if isinstance(results['field'], pd.DataFrame):
            plt.figure(figsize=(12, 8))  # Wide figure
            
            if results['field'].index.name == 'ur_bin':
                # Already indexed properly
                plot_df = results['field']
            elif 'ur_bin' in results['field'].columns:
                # Need to set index
                plot_df = results['field'].set_index('ur_bin')
            else:
                # Can't determine proper structure
                print("Cannot identify proper structure for field data")
                return
                
            # Plot the data
            plot_df.plot(kind='bar', stacked=True)
            
#             plt.title('Field of Study Composition by Initial Unemployment Rate', fontsize=14)
            plt.ylabel('Proportion', fontsize=12)
            plt.xlabel('Unemployment Rate Bin', fontsize=10)
            
            # Make x-axis labels horizontal
            plt.xticks(rotation=0)
            
            # Add a bit more padding between x-axis labels and legend
            plt.subplots_adjust(bottom=0.3)
            
            # Create descriptive labels for legend
            legend_labels = []
            for col in plot_df.columns:
                if col in field_labels:
                    legend_labels.append(field_labels[col])
                else:
                    legend_labels.append(f'Field {col}')
            
            # Move legend to the bottom with descriptive labels and smaller font - further down
            plt.legend(
                title='Field of Study',
                labels=legend_labels,
                bbox_to_anchor=(0.5, -0.25),  # Move legend further down
                loc='upper center',  # Anchor at upper center
                fontsize=7,  # Smaller font size
                title_fontsize=10,  # Smaller title font size
                ncol=4  # Use multiple columns for the legend
            )
            
            plt.tight_layout()
            
            plt.savefig('field_composition.png', dpi=300)
            plt.show()
        else:
            print("Field data is not a DataFrame")
            
#     # 5. Time Gap Plot
#     if 'gap' in results:
#         plt.figure(figsize=(10, 6))
#         results['gap']['mean'].plot(kind='bar')
#         plt.title('Average Time Between Bachelor\'s and Master\'s by Initial Unemployment Rate')
#         plt.ylabel('Years')
#         plt.xticks(rotation=0)  # Horizontal labels
#         plt.tight_layout()
#         plt.savefig('masters_timing_analysis.png', dpi=300)
#         plt.show()

# Run the analysis
results, masters_df = analyze_masters_composition(df)

# Create visualizations
print("\n--- GENERATING VISUALIZATIONS ---")
visualize_composition_results(results)
print("Visualizations completed.")


In [ ]:
# Run the analysis
results, masters_df = analyze_masters_composition(df)

# Print all results in a readable format
print("\n==== MASTERS COMPOSITION ANALYSIS RESULTS ====\n")

# Print the number of masters degree holders
print(f"Total number of master's degree holders: {len(masters_df)}")

# Print gender composition results
print("\n--- GENDER COMPOSITION ---")
if 'gender' in results:
    print("Proportion female by unemployment rate bin:")
    print(results['gender'])
    
    print("\nRegression of gender on unemployment rate:")
    print(results['gender_reg'].summary())  # Print only coefficient table

# Print race composition results
print("\n--- RACIAL COMPOSITION ---")
if 'race' in results:
    print("Proportion by race and unemployment rate bin:")
    print(results['race'])
    
    print("\nRegressions of racial indicators on unemployment rate:")
    for race_col, reg in results.get('race_regs', {}).items():
        print(f"\n{race_col}:")
        print(reg.summary().tables[1])  # Print only coefficient table

# Print age results
print("\n--- AGE DISTRIBUTION ---")
if 'age' in results:
    print("Age statistics by unemployment rate bin:")
    print(results['age'])
    
    print("\nRegression of age on unemployment rate:")
    print(results['age_reg'].summary().tables[1])  # Print only coefficient table

# Print field of study results
print("\n--- FIELD OF STUDY COMPOSITION ---")
if 'field' in results:
    print("Field proportions by unemployment rate bin:")
    print(results['field'].head(20))  # Show first 20 rows to avoid overwhelming output
    
    print("\nRegressions of field indicators on unemployment rate:")
    print("Note: Only showing first 5 fields...")
    count = 0
    for field_col, reg in list(results.get('field_regs', {}).items())[:5]:  # Show only first 5 to avoid too much output
        print(f"\n{field_col}:")
        print(reg.summary().tables[1])  # Print only coefficient table
        count += 1
    
    remaining = len(results.get('field_regs', {})) - count
    if remaining > 0:
        print(f"\n... and {remaining} more field regressions (not shown)")

# # Try visualizing results
# visualize_composition_results(results)
# visualize_composition_results(results)
# regression_results = run_direct_regression_analysis(df)

### Test other cohort trend definitions

In [ ]:
import pandas as pd
# import numpy as np
# import statsmodels.formula.api as smf
# from statsmodels.discrete.discrete_model import Probit

# def prepare_data(df):
#     """
#     Prepare data for regression analysis
#     """
#     # Keep only the last observation for each person
#     df = df.sort_values('REFYR').drop_duplicates(subset='assigned_id', keep='last')
#     # Create immediate masters indicator
#     df['immediate_masters'] = ((df['DGRDG'] == 2) & 
#                              (df['DGRYR'] - df['BAYR'] <= 3)).astype(int)
#     df['with_masters'] = (df['DGRDG'] == 2 ).astype(int)
    
#     # Create cohort trend (centered)
#     df['cohort_trend'] = (df['BAYR'] - df['BAYR'].mean()) / df['BAYR'].std()
#     df['cohort_trend_sq'] = df['cohort_trend']**2       # Quadratic trend
#     df['cohort_trend_cub'] = df['cohort_trend']**3       # Quadratic trend
    
#     # Create dummy variables for categorical variables
#     cohort_dummies = pd.get_dummies(df['BAYR'], prefix='cohort')
#     gender_dummies = pd.get_dummies(df['GENDER'], prefix='gender')
#     race_dummies = pd.get_dummies(df['RACEM'], prefix='race')
#     major_dummies = pd.get_dummies(df['NBAMEMG'], prefix='major')
    
#     # Print diagnostic information
#     print("Sample information:")
#     print(f"Number of unique individuals: {df['assigned_id'].nunique()}")
#     print(f"Year range: {df['REFYR'].min()} - {df['REFYR'].max()}")
#     print(f"Bachelor graduation year range: {df['BAYR'].min()} - {df['BAYR'].max()}")
#     print("\nDistribution of immediate_masters:")
#     print(df['immediate_masters'].value_counts(normalize=True))
#     print("\nUnemployment rate summary:")
#     print(df['gradur_ba'].describe())
    
#     return df

# def run_probit_analysis(df, formula):
#     """
#     Run probit regression using formula interface
#     """
#     model = smf.probit(formula, data=df)
#     results = model.fit(cov_type='HC1')
    
#     return results

# def run_probit_analysis_robust_cohort(df, formula):
#     """
#     Run probit regression using formula interface
#     """
    
#     model = smf.probit(formula, data=df)
#    # Cluster standard errors at the graduation cohort level
#     results = model.fit(cov_type='cluster', 
#                        cov_kwds={'groups': df['BAYR']})
    
#     return results


# # To use:
# df = prepare_data(df_final)


formula_w_cohort_trends = '''
with_masters ~ UR + C(BAYR)
'''
# formula_w_cohort_trends = '''
# with_masters ~ UR + cohort_trend + cohort_trend_sq
# '''
results_w_cohort_trends = run_probit_analysis(df, formula_w_cohort_trends)
print(results_w_cohort_trends.summary())

results_w_cohort_trends_1 = run_probit_analysis_robust_cohort(df, formula_w_cohort_trends)
print(results_w_cohort_trends_1.summary())
